# Notebook 08: Failure Prediction Model Development & Experiments

**Project:** AI Operations Intelligence Platform  
**Phase:** Week 1 Day 6 (Failure Risk Model)  
**Data Scope:** AI4I 2020 Dataset (Training Partition: 8,000 rows only)

---

## Governance Rules:
- Rule R6 (Pre-Registration): Search grid and selection rule fixed.
- Rule R3/R4 (No Leakage): twf, hdf, pwf, osf, rnf, udi, product_id excluded.
- Rule R2 (Test Discipline): Test partition is never loaded.
- Rule R12 (Clean Notebook): Outputs stripped.


In [ ]:
# 0. Setup and Environment Verification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.prediction.config import SEED, EXPERIMENTS_LOG_PATH, OOF_PREDICTIONS_PATH
from ml.prediction.data import load_train
from ml.prediction.features import build_features
from ml.prediction.calibration import choose_calibrator
from ml.prediction.thresholds import compute_thresholds, compute_category_reliability
from ml.prediction.controls import (
    assert_forbidden_columns_absent,
    run_feature_leakage_scan,
)
from ml.prediction.model import FailureModel
from ml.eda.plotting import apply_style

apply_style()
print(f'Random seed set to {SEED}')


## 1. Training Partition Inspection
We verify that the training partition contains 8,000 rows and inspect failure prevalence.

In [ ]:
# 1. Load Training Partition
train_df = load_train()
n_total = len(train_df)
n_failures = int(train_df['machine_failure'].sum())
prevalence = n_failures / n_total
print(f'Training Rows: {n_total:,}')
print(f'Failures: {n_failures} ({prevalence:.2%})')


## 2. Experiments Log & Ablations Audit
We examine failure_experiments.csv comparing XGBoost against ablations and baselines.

In [ ]:
# 2. Experiments Log Summary
exp_df = pd.read_csv(EXPERIMENTS_LOG_PATH)
print(f'Total experiments recorded: {len(exp_df)}')
summary = exp_df[exp_df['config_id'].isin([
    'XGB-PHYS-08', 'ABL-RAW-01', 'BASE-LOGREG-01', 'BASE-MAJORITY-01'
])]
cols = ['config_id', 'model_type', 'feature_set', 'mean_pr_auc', 'mean_roc_auc', 'mean_brier']
print(summary[cols].to_string(index=False))


## 3. Calibration & Risk Thresholds
We audit the chosen calibrator (isotonic) and derived operating risk thresholds.

In [ ]:
# 3. Calibration & Risk Categories
oof_df = pd.read_parquet(OOF_PREDICTIONS_PATH)
rep1 = oof_df[oof_df['repeat'] == 1].reset_index(drop=True)
_, cal = choose_calibrator(
    rep1['y_prob'].to_numpy(), rep1['y_true'].to_numpy(), rep1['fold'].to_numpy()
)
cal_p = cal.predict_proba(rep1['y_prob'].to_numpy())
thresh = compute_thresholds(rep1['y_true'].to_numpy(), cal_p)
print(f'Thresholds: {thresh.to_dict()}')
rel = compute_category_reliability(rep1['y_true'].to_numpy(), cal_p, thresh)
print(rel.to_string(index=False))


## 4. Negative Controls (Rule R8)
Audit feature leakage scan and confirm zero forbidden columns exist in feature matrix.

In [ ]:
# 4. Negative Controls
X_train = build_features(train_df, feature_set='physics')
assert_forbidden_columns_absent(X_train)
print('Forbidden columns check: PASSED (zero leakage features).')
scan = run_feature_leakage_scan(X_train, train_df['machine_failure'].to_numpy())
print(scan.to_string(index=False))
